[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/weaviate/recipes/blob/main/weaviate-features/drop-vector-index/drop-vector-index.ipynb)

# Drop a Vector Index in Weaviate

*Weaviate version `1.40.0` · Weaviate Python client `4.23.1` · OpenAI `text-embedding-3-large` and Cohere `embed-multilingual-v3.0` vectors — by Duda Nogueira ([Weaviate](https://weaviate.io))*

Collections with [named vectors](https://docs.weaviate.io/weaviate/config-refs/collections#named-vectors) tend to grow over time. You add a new embedding model, run it side by side with the old one, switch your queries over... and the old vector index keeps eating disk and memory.

Since **Weaviate v1.40** (GA), you can **drop the index of a named vector** from an existing collection, without recreating the collection or re-ingesting your data. The objects stay, the other vectors stay, and the disk space used by the dropped vector is reclaimed.

In this recipe we play out a typical model migration:

1. A collection of Wikipedia passages has two named vectors: `openai_vec` (the model we are moving away from) and `cohere_vec` (the model we are moving to)
2. We measure how much disk each vector uses
3. We drop `openai_vec` and watch the drop complete
4. We confirm the disk was reclaimed and that search on `cohere_vec` still works

> ⚠️ **Dropping a vector index is irreversible.** The index is deleted from disk and the vector is removed from every object in the collection. Make sure you have a [backup](https://docs.weaviate.io/deploy/configuration/backups) if you might need it again.

In [1]:
!pip install -Uq "weaviate-client>=4.23.1" datasets

## 1. Start an embedded Weaviate

We use [Embedded Weaviate](https://docs.weaviate.io/deploy/installation-guides/embedded) so this notebook runs anywhere, including Colab. We store its data in a local folder so we can measure it.

In [2]:
import shutil
import weaviate

DATA_PATH = "./weaviate-data"
shutil.rmtree(DATA_PATH, ignore_errors=True)

client = weaviate.connect_to_embedded(
    version="1.40.0",
    persistence_data_path=DATA_PATH,
    environment_variables={"LOG_LEVEL": "error"},
)

print("Weaviate version:", client.get_meta()["version"])

Weaviate version: 1.40.0


## 2. Load pre-vectorized data

[`weaviate/wiki-sample`](https://huggingface.co/datasets/weaviate/wiki-sample) has the same Wikipedia passages embedded with several models, in the same order. We stream 2,000 passages twice: once with OpenAI `text-embedding-3-large` vectors (3072 dimensions) and once with Cohere `embed-multilingual-v3.0` vectors (1024 dimensions).

Because the vectors are already computed, this recipe needs **no API keys**.

In [3]:
from itertools import islice
from datasets import load_dataset

N = 2000


def load(config):
    return list(islice(load_dataset("weaviate/wiki-sample", config, split="train", streaming=True), N))


openai_rows = load("openai-text-embedding-3-large")
cohere_rows = load("cohere-embed-multilingual-v3")

assert [r["wiki_id"] for r in openai_rows] == [r["wiki_id"] for r in cohere_rows]
print(f"Loaded {N} passages | OpenAI dims: {len(openai_rows[0]['vector'])} | Cohere dims: {len(cohere_rows[0]['vector'])}")

Loaded 2000 passages | OpenAI dims: 3072 | Cohere dims: 1024


## 3. Create a collection with two named vectors

Each named vector is configured with the vectorizer that produced its embeddings. Since we provide the vectors at import, Weaviate doesn't call OpenAI or Cohere.

In [4]:
from weaviate.classes.config import Configure, DataType, Property

client.collections.delete("Passage")
passages = client.collections.create(
    "Passage",
    properties=[
        Property(name="title", data_type=DataType.TEXT),
        Property(name="text", data_type=DataType.TEXT),
    ],
    vector_config=[
        Configure.Vectors.text2vec_openai(name="openai_vec", model="text-embedding-3-large"),
        Configure.Vectors.text2vec_cohere(name="cohere_vec", model="embed-multilingual-v3.0"),
    ],
)

with passages.batch.fixed_size(batch_size=200) as batch:
    for o, c in zip(openai_rows, cohere_rows):
        batch.add_object(
            properties={"title": c["title"], "text": c["text"]},
            vector={"openai_vec": o["vector"], "cohere_vec": c["vector"]},
        )

print("Failed objects:", len(passages.batch.failed_objects))
print("Objects in collection:", passages.aggregate.over_all().total_count)

Failed objects: 0
Objects in collection: 2000


Both vectors are searchable. We use the stored vectors of a passage as the query, so no provider call is needed:

In [5]:
query_index = next(i for i, r in enumerate(cohere_rows) if r["title"] == "Telescope")
print("Query passage:", cohere_rows[query_index]["title"], "\n")

for name, rows in [("openai_vec", openai_rows), ("cohere_vec", cohere_rows)]:
    response = passages.query.near_vector(rows[query_index]["vector"], target_vector=name, limit=4)
    print(name, "->", [o.properties["title"] for o in response.objects])

Query passage: Telescope 

openai_vec -> ['Telescope', 'William Herschel', 'History of astronomy', 'Orion complex']
cohere_vec -> ['Telescope', 'Special relativity', 'William Herschel', 'Orion complex']


## 4. Measure disk usage

Each shard of a collection lives in its own folder. Inside it, every named vector gets its own vector index files (`vectors_<name>...`), and the objects, including their vectors, are stored in the `lsm` folder.

This helper sums the size of each item in the shard folder.

In [6]:
import os


def folder_size(path):
    return sum(
        os.path.getsize(os.path.join(root, f))
        for root, _, files in os.walk(path)
        for f in files
    )


def shard_usage(collection_name):
    collection_path = os.path.join(DATA_PATH, collection_name.lower())
    shard = os.listdir(collection_path)[0]  # single node, single shard
    shard_path = os.path.join(collection_path, shard)
    usage = {item: folder_size(os.path.join(shard_path, item)) for item in os.listdir(shard_path)}
    return {k: v for k, v in sorted(usage.items(), key=lambda kv: -kv[1]) if v > 0}


def print_usage(usage):
    for item, size in usage.items():
        print(f"{size / 1e6:8.1f} MB  {item}")
    print(f"{sum(usage.values()) / 1e6:8.1f} MB  TOTAL")


before = shard_usage("Passage")
print_usage(before)

    41.8 MB  lsm
     0.8 MB  vectors_cohere_vec.hnsw.commitlog.d
     0.7 MB  vectors_openai_vec.hnsw.commitlog.d
    43.3 MB  TOTAL


With 3072 dimensions, each OpenAI vector is three times the size of a Cohere one. Most of that space is in the `lsm` folder, where the objects store their vectors. The HNSW commit logs hold the graph structure, which is small for 2,000 objects.

## 5. Drop the `openai_vec` index

The drop is a single REST call: `DELETE /v1/schema/{collection}/vectors/{vector}/index`.

> 💡 Native support (`collection.config.delete_vector_index("openai_vec")`) is already merged into the Python client and ships in the release after `4.23.1`. Until then, we call the endpoint directly.

The call returns as soon as Weaviate **accepts** the request. The cleanup runs in the background: the vector first shows with index type `none` in the schema, then disappears once the drop has completed.

In [7]:
import requests

BASE_URL = "http://localhost:8079/v1"  # Embedded Weaviate listens on port 8079

response = requests.delete(f"{BASE_URL}/schema/Passage/vectors/openai_vec/index")
print(response.status_code)

200


Let's poll the schema until `openai_vec` is gone.

> Here we read the schema through REST too: with Python client `4.23.1`, `collection.config.get()` raises an error while a drop is in progress. This is fixed in the next client release.

In [8]:
import time


def vector_index_types(collection_name):
    schema = requests.get(f"{BASE_URL}/schema/{collection_name}").json()
    return {name: cfg.get("vectorIndexType") for name, cfg in schema["vectorConfig"].items()}


start = time.time()
while True:
    index_types = vector_index_types("Passage")
    print(f"{time.time() - start:5.0f}s  {index_types}")
    if "openai_vec" not in index_types:
        break
    time.sleep(5)

print(f"Drop completed in about {time.time() - start:.0f}s")

    0s  {'cohere_vec': 'hnsw', 'openai_vec': 'none'}


    5s  {'cohere_vec': 'hnsw', 'openai_vec': 'none'}


   10s  {'cohere_vec': 'hnsw', 'openai_vec': 'none'}


   15s  {'cohere_vec': 'hnsw', 'openai_vec': 'none'}


   20s  {'cohere_vec': 'hnsw', 'openai_vec': 'none'}


   25s  {'cohere_vec': 'hnsw', 'openai_vec': 'none'}


   30s  {'cohere_vec': 'hnsw', 'openai_vec': 'none'}


   35s  {'cohere_vec': 'hnsw'}
Drop completed in about 35s


## 6. Check the results

The disk space is back:

In [9]:
after = shard_usage("Passage")
print_usage(after)

saved = sum(before.values()) - sum(after.values())
print(f"\nReclaimed {saved / 1e6:.1f} MB ({saved / sum(before.values()):.0%} of the shard)")

    17.3 MB  lsm
     0.8 MB  vectors_cohere_vec.hnsw.commitlog.d
    18.1 MB  TOTAL

Reclaimed 25.1 MB (58% of the shard)


The objects are all still there, and `cohere_vec` search works as before:

In [10]:
print("Objects in collection:", passages.aggregate.over_all().total_count)

response = passages.query.near_vector(cohere_rows[query_index]["vector"], target_vector="cohere_vec", limit=4)
print("cohere_vec ->", [o.properties["title"] for o in response.objects])

Objects in collection: 2000
cohere_vec -> ['Telescope', 'Special relativity', 'William Herschel', 'Orion complex']


The objects no longer carry the dropped vector, and searching it is rejected:

In [11]:
obj = passages.query.fetch_objects(limit=1, include_vector=True).objects[0]
print("Vectors on the object:", list(obj.vector.keys()))

try:
    passages.query.near_vector(openai_rows[query_index]["vector"], target_vector="openai_vec", limit=1)
except Exception as e:
    print("Search on openai_vec failed as expected:\n", e)

Vectors on the object: ['cohere_vec']
Search on openai_vec failed as expected:
 Query call with protocol GRPC search failed with message extract target vectors: class Passage does not have named vector openai_vec configured. Available named vectors [cohere_vec].


## Good to know

- **Only named vectors** can be dropped. A collection's legacy (unnamed) vector can't.
- **It's asynchronous.** A successful call means the drop was accepted, not finished. Repeating the call while a drop is in progress is safe.
- **Re-adding is possible.** Once the drop has completed, you can add a new, empty vector with the same name with `collection.config.add_vector()`. You then need to re-ingest its embeddings.
- **It works with every index type** (HNSW, flat, dynamic, HFresh, and MUVERA multi-vector), and with multi-tenancy, including inactive (cold) tenants.
- **RBAC**: dropping a vector index rewrites every object in the collection, so it needs `update_collections` on the collection (data and metadata), not just metadata access.

Learn more in the [Weaviate docs](https://docs.weaviate.io/weaviate/manage-collections/vector-config).

In [12]:
client.close()